# Péndulo simple II

> Tu colega nunca ha visto un péndulo en el que la aproximación de angulos pequeños no
> sea suficiente y quiere ver las diferencias en el movimiento.
> Compara el movimiento del péndulo real con el del péndulo aproximado
> considerando las mismas condiciones del ejericio anterior.

Construyendo sobre lo que vimos en el ejemplo de [oscilador armońico](05_hooke.ipynb)
y en el [primer ejercicio de péndulo simple](06_pendulum.ipynb), la aproximación de
ángulos pequeños resulta en una ecuación de posición
$$
  \phi(t) \approx \phi_{\max} \cos{ \left( \sqrt{\frac{g}{L}} t \right) },
$$
y esperamos que esta ecuación falle para ángulos grandes.

Ahora queremos encontrar la
función de posición contra tiempo sin recurrir a aproximaciones.
La ecuación diferencial ordinaria (EDO) completa a resolver es
$$
  \ddot{\phi}(t) = - \frac{g}{L} \sin{\phi(t)}.
$$
Al ser de segundo orden, hacer una discretización directa no es muy cómodo.
En su lugar, convirtamos esta EDO de segundo orden en un sistema de dos EDOs de
primer orden.

Si definimos
\begin{gather*}
  v_{\phi} = \dot{\phi}, \\
  a_{\phi} = \dot{v}_{\phi} = \ddot{\phi}
\end{gather*}
entonces podemos escribir

$$
  \frac{\mathrm{d}}{\mathrm{d} t}
  \begin{pmatrix}
    v_{\phi} \\
    \phi
  \end{pmatrix}
  =
  \begin{pmatrix}
    a_{\phi} \\
    v_{\phi}
  \end{pmatrix}
  =
  \begin{pmatrix}
    -(g/L) \sin{\phi(t)} \\
    v_{\phi}(t)
  \end{pmatrix}.
$$

La discretización simple del sistema es entonces

$$
  \begin{pmatrix}
    v_{\phi}(t + {\Delta t}) \\
    \phi(t + {\Delta t})
  \end{pmatrix}
  \approx
  \begin{pmatrix}
    v_{\phi}(t ) \\
    \phi(t)
  \end{pmatrix}
  +
  \begin{pmatrix}
    -(g/L) \sin{\phi(t)} \\
    v_{\phi}(t)
  \end{pmatrix}
  {\Delta t}
$$

Esto es el _método de Euler_ aplicado a una función $\mathbb{R} \to \mathbb{R}^2$.
Es muy parecido al método de sumas de Riemann para encontrar integrales
(naturalmente, pues estamos integrando la ecuación diferencial).
La diferencia es que en este caso queremos todos los valores intermedios de la
integración, no solo el valor final.

Esta separación de posición y velocidad es muy natural;
recordemos de ecuaciones diferenciales elementales que
para resolver un problema de valor inicial de segundo orden se requieren dos condiciones
iniciales, $f(0)$ y $f'(0)$. Similarmente, para encontrar la evolución del sistema
requerimos posición inicial y velocidad inicial.

In [ ]:
import math

In [ ]:
g = 9.80665  # m/s^2
# m = 1  [kg] (no se usa)
L = 30  # m
phi_max = math.radians(85)

Implementemos el método de Euler.

In [ ]:
def euler_integrate(dfunc, yi, vi, tstart, tend, N):
    """
    Integra una función usando el método de Euler.

    Parámetros
    ----------
    dfunc: derivada de la función
            `dfunc(t, y, v)`
    yi: valor inicial de la función
    vi: valor inicial de la derivada de la función
    tstart, tend: intervalo de integración
    N:  número de nodos

    Regresa
    -------
    t:  arreglo de puntos de evaluación
    [v, y]: arreglo con valores encontrados
    """

    def euler_step(h, t, y, v):
        """
        Calcula

          Delta y = (dy/dx) {Delta x}

        Parámetros
        ----------
        h:  tamaño del paso de integración
        t:  valor de t al integrar
        y:  valor de y al integrar
        v:  valor de v al integrar

        Regresa
        -------
        Dy: cambio de y
        """

        Dv = dfunc(t, y, v) * h
        Dy = v * h
        return Dv, Dy

    def euler_array():
        h = (tend - tstart) / N
        tvals = [tstart + j * h for j in range(0, N + 1)]
        results_v = [vi]
        results_y = [yi]
        for j in range(1, N + 1):
            Dv, Dy = euler_step(h, tvals[j - 1], results_y[j - 1], results_v[j - 1])
            results_v.append(results_v[j - 1] + Dv)
            results_y.append(results_y[j - 1] + Dy)

        return tvals, results_y, results_v

    return euler_array()

:::{note} Observaciones
- Nota que constantemente estamos alocando nuevos elementos a listas. Esto requiere
  aumentar dinámicamente el tamaño de la lista, lo que es *muy computacionalmente pesado*
  (_i.e._ lento).

- Imagina qué fácil sería leer esta implementación y fuera posible sumar listas como si
  fueran matrices:
  ```python
  [a, b] + [c, d] = [a, b, c, d] != [a + c, b + d]
                  # ^ esto ocurre   ^ esto harían vectores o matrices
  ```
  Hablaremos más sobre matrices pronto.
:::

:::{warning} ¡No uses este código en producción!
No lo copies y pegues en tus proyectos.
Escribí estas funciones con fin ilustrativo.
Son lentas y no tienen mecanismos para tratar errores o estimar errores absolutos.
:::

De paso, implementemos la solución de ángulos pequeños para comparar:

In [ ]:
def pendulum_smallangle(t, L=L, phi_max=phi_max):
    """
    Implementación simple del péndulo con ángulos pequeños.
    """
    return -phi_max * math.cos(math.sqrt(g / L) * t)

e importemos la solución exacta de `pendulum.py`
(de nuevo, la implementación exacta no importa mucho por ahora):

In [ ]:
import pendulum as pd

In [ ]:
help(pd.movement)

La manera más fácil de comparar será visualmente, así que graficaremos nuestros
resultados usando `matplotlib`.
No te preocupes por cómo graficar por ahora, lo veremos después.

In [ ]:
import matplotlib.pyplot as plt


def pendulum_graphic_comparison(
    tstart, tend, N=100, g=g, L=L, phi_max=phi_max, show_smallangle=True
):

    time, phi, vphi = euler_integrate(
        lambda t, phi, v: -(g / L) * math.sin(phi), -phi_max, 0.0, tstart, tend, N
    )

    fig, ax = plt.subplots(layout="constrained")

    ax.set_xlabel(r"$t$ [s]")
    ax.set_ylabel(r"$\phi$ [rad]")

    ax.plot(
        time,
        phi,
        label="Péndulo numérico",
    )
    ax.plot(
        time,
        [-pd.movement(t, L=L, g=g, phi_max=phi_max) for t in time],
        label="Solución exacta",
        color="0.5",
        ls="dashed",
    )

    if show_smallangle:
        ax.plot(
            time,
            [pendulum_smallangle(t) for t in time],
            label="Péndulo de ángulos pequeños",
        )

    ax.legend(loc="best")

In [ ]:
pendulum_graphic_comparison(0, 30)

_Vaya, esto no sirve para nada_, podrías estar pensando, pero la razón de este
comportamiento es muy simple. Nota que nuestro integrador usa `N = 100` por defecto.
Incrementemos esta cantidad.

In [ ]:
pendulum_graphic_comparison(0, 30, N=1000)

_Ah_, mucho mejor. Vimos que si nuestra partición no tiene suficientes puntos
(i.e. si están muy separados)
el error en la discretización se vuelve demasiado grande. El costo de mejor precisión
es más operaciones
(que se refleja en un tiempo de ejecución mayor).

Ahora podemos ver lo inadecuada que es la aproximación de ángulos pequeños para nuestro
sistema.
Además, vemos que nuestra implementación numérica sigue de cerca el resultado real en
este intervalo. Sin embargo, si extendemos la zona de interés...

In [ ]:
pendulum_graphic_comparison(0, 90, N=3000, show_smallangle=False)

...vemos que la velocidad del integrador aumenta en cada oscilación.

Este es un efecto común en integración con oscilaciones. El método de Euler tiende a
mandar la solución _por encima_ de su amplitud regular, alejándose más del centro en
cada punto extremo.

In [ ]:
pendulum_graphic_comparison(0, 90, N=30000, show_smallangle=False)

¿Y en el ejercicio anterior? Nota que en ese caso las evaluaciones no dependen
de valores obtenidos con anterioridad, mientras que aquí sí. Los errores se propagan
hacia adelante en nuestras soluciones.

No hay de qué preocuparse; esta clase de problemas y más ya están solucionados.
Los integradores de uso profesional implementan algoritmos que reducen estos efectos.